In [ ]:
#Filtering Nature & Environment Category articles from the combined dataset

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("/content/drive/MyDrive/Venice_RC15/Datasets/News/combined_news_categorized.csv")

# -----------------------------
# Filter rows containing "Nature & environment"
# -----------------------------
# Some rows have multiple categories separated by commas
df_filtered = df[df['category'].str.split(',').apply(lambda x: 'Nature & environment' in [i.strip() for i in x])]

# -----------------------------
# Export filtered CSV
# -----------------------------
output_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news.csv"
df_filtered.to_csv(output_path, index=False)

print(f"✅ Filtered CSV exported: {output_path}")


In [ ]:
#Extracting the content of the articles in the Nature & Environment category to perform sentiment analysis on them.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# Load CSV
df = pd.read_csv("/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news.csv")

# Function to extract text from a URL
def extract_article_text(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return None
        soup = BeautifulSoup(response.content, "html.parser")

        # Extract all paragraphs
        paragraphs = soup.find_all('p')
        text = ' '.join([p.get_text() for p in paragraphs])
        return text.strip()
    except Exception as e:
        return None

# -----------------------------
# Add content column
# -----------------------------
tqdm.pandas()  # For progress bar
df['content'] = df['link'].progress_apply(extract_article_text)

# -----------------------------
# Save new CSV with content
# -----------------------------
output_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news_with_content.csv"
df.to_csv(output_path, index=False)

print(f"✅ CSV with article content saved to: {output_path}")


#Running Sentiment Analysis

In [ ]:
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import re

# -----------------------------
# Download VADER lexicon if not already
# -----------------------------
nltk.download('vader_lexicon')

# -----------------------------
# Load CSV
# -----------------------------
df = pd.read_csv("/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news_with_content.csv")

# -----------------------------
# Initialize VADER
# -----------------------------
sid = SentimentIntensityAnalyzer()

# -----------------------------
# Function to compute sentiment (-1 to 1)
# -----------------------------
def get_sentiment(text):
    if pd.isna(text):
        return 0  # Neutral for empty content
    score = sid.polarity_scores(text)['compound']  # compound score is in [-1, 1]
    return score

# Combine title and content
df['text_combined'] = df['title'].fillna('') + ' ' + df['content'].fillna('')

# Apply sentiment
df['sentiment'] = df['text_combined'].apply(get_sentiment)

# -----------------------------
# Compute topic intensity (0 to 1)
# -----------------------------
keywords = ["Water", "High Tide", "Climate Change", "Mose", "Flood", "Flooding",
            "Environment", "Lagoon", "Sea", "Green", "Canal", "Biodiversity",
            "Animals", "Ecology", "Nature", "Acqua Alta", "Natura"]

# Precompile regex for faster matching (case-insensitive)
pattern = re.compile(r'\b(' + '|'.join(keywords) + r')\b', re.IGNORECASE)

def topic_intensity(text):
    if pd.isna(text) or text.strip() == '':
        return 0
    matches = pattern.findall(text)
    # Normalize by length of text (number of words)
    num_words = len(text.split())
    intensity = len(matches) / num_words if num_words > 0 else 0
    return min(intensity, 1)  # Cap at 1

df['topic_intensity'] = df['text_combined'].apply(topic_intensity)

# -----------------------------
# Save new CSV
# -----------------------------
output_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news_sentiment_topic.csv"
df.to_csv(output_path, index=False)

print(f"✅ CSV with sentiment and topic intensity saved: {output_path}")


In [ ]:
#Checking for topic intensity in the category

In [ ]:
import pandas as pd

# Load CSV
csv_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news_sentiment_topic.csv"
df = pd.read_csv(csv_path)

# -----------------------------
# Normalize topic_intensity
# -----------------------------
max_intensity = df['topic_intensity'].max()
if max_intensity > 0:
    df['topic_intensity_normalized'] = df['topic_intensity'] / max_intensity
else:
    df['topic_intensity_normalized'] = df['topic_intensity']  # All zeros, keep as is

# -----------------------------
# Save back to CSV
# -----------------------------
output_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news_sentiment_topic_normalized.csv"
df.to_csv(output_path, index=False)

print(f"✅ Topic intensity normalized and saved to: {output_path}")


#Exporting clean csv with sentiment analysis and topic intensity

In [ ]:
import pandas as pd

# Load CSV
csv_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/Nature_Environment_news_sentiment_topic.csv"
df = pd.read_csv(csv_path)

# Select only sentiment and topic_intensity columns
df_subset = df[['sentiment', 'topic_intensity']]

# Export to new CSV
output_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/sentimentNintensity.csv"
df_subset.to_csv(output_path, index=False)

print(f"✅ Sentiment and topic_intensity exported to: {output_path}")


#Normalising topic intensity values for rhino visualization

In [ ]:
import pandas as pd

# Load CSV
csv_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/sentimentNintensity.csv"
df = pd.read_csv(csv_path)

# Multiply topic_intensity by 19.19827983
df['topic_intensity'] = df['topic_intensity'] * 40.19827983

# Save back to CSV (overwrite or as a new file)
output_path = "/content/drive/MyDrive/Venice_RC15/Datasets/News/sentimentNintensity_scaled.csv"
df.to_csv(output_path, index=False)

print(f"✅ topic_intensity multiplied and saved to: {output_path}")
